# 03 — Reward Model

Trains a reward model on human preference pairs using TRL's `RewardTrainer`.
The reward model is a `distilgpt2` head adapted for scalar scoring —
it will score generated bios during PPO in `04_ppo.ipynb`.

Requires: preference data splits from `02_preference_data.ipynb`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jonasneves/aipi590-challenge-2/blob/main/notebooks/03_reward_model.ipynb)

In [ ]:
# Setup — install dependencies, clone repo, configure environment
!pip install -q trl peft transformers datasets accelerate bitsandbytes

import os
from google.colab import userdata
os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")

import sys
sys.path.insert(0, "/content/aipi590-challenge-2")

from src.colab_utils import prepare_notebook, publish_artifacts

repo_root, paths = prepare_notebook("aipi590-challenge-2")
print(f"Repo root: {repo_root}")

# Verify SFT checkpoint exists
sft_checkpoint = paths["results"] / "sft_checkpoint"
if not sft_checkpoint.exists():
    print(f"WARNING: SFT checkpoint not found at {sft_checkpoint}")
    print("Falling back to base distilgpt2 for reward model initialization.")
    REWARD_BASE = "distilgpt2"
else:
    REWARD_BASE = str(sft_checkpoint)
    print(f"Using SFT checkpoint: {REWARD_BASE}")

In [ ]:
# Load preference dataset
import sys
sys.path.insert(0, str(repo_root))

from src.dataset import load_preference_dataset

train_path = paths["data"] / "preference_train.jsonl"
val_path   = paths["data"] / "preference_val.jsonl"

if not train_path.exists():
    raise FileNotFoundError(
        f"Preference data not found at {train_path}. "
        "Run 02_preference_data.ipynb first."
    )

# Combine for load_preference_dataset (it handles the split internally)
# Or load train/val separately from the pre-split files
from datasets import Dataset
import json

def load_jsonl_as_pref_dataset(path):
    from src.dataset import format_sft_prompt
    records = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    formatted = []
    for r in records:
        if r.get("chosen") not in ("A", "B"):
            continue
        prompt = format_sft_prompt(r["person"])
        chosen_text   = r["a"] if r["chosen"] == "A" else r["b"]
        rejected_text = r["b"] if r["chosen"] == "A" else r["a"]
        formatted.append({"prompt": prompt, "chosen": chosen_text, "rejected": rejected_text})
    return Dataset.from_dict({
        "prompt":   [r["prompt"]   for r in formatted],
        "chosen":   [r["chosen"]   for r in formatted],
        "rejected": [r["rejected"] for r in formatted],
    })

train_dataset = load_jsonl_as_pref_dataset(train_path)
eval_dataset  = load_jsonl_as_pref_dataset(val_path)

print(f"Train: {len(train_dataset)} pairs")
print(f"Eval:  {len(eval_dataset)} pairs")
print("\nSample:")
print("  prompt:   ", train_dataset[0]["prompt"][:80], "...")
print("  chosen:   ", train_dataset[0]["chosen"][:80])
print("  rejected: ", train_dataset[0]["rejected"][:80])

In [ ]:
# Build and train the reward model
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from trl import RewardTrainer, RewardConfig

tokenizer = AutoTokenizer.from_pretrained(REWARD_BASE)
tokenizer.pad_token = tokenizer.eos_token

reward_model = AutoModelForSequenceClassification.from_pretrained(
    REWARD_BASE,
    num_labels=1,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
reward_model.config.pad_token_id = tokenizer.pad_token_id

reward_config = RewardConfig(
    output_dir="/content/reward_output",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    max_length=256,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = RewardTrainer(
    model=reward_model,
    args=reward_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
# Plot reward score distribution on eval set
import torch
import matplotlib.pyplot as plt
import numpy as np

reward_model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
reward_model = reward_model.to(device)

chosen_scores = []
rejected_scores = []

for i in range(min(len(eval_dataset), 50)):
    row = eval_dataset[i]
    for text, store in [(row["chosen"], chosen_scores), (row["rejected"], rejected_scores)]:
        full_text = row["prompt"] + text
        enc = tokenizer(
            full_text,
            return_tensors="pt",
            truncation=True,
            max_length=256,
            padding=True,
        ).to(device)
        with torch.no_grad():
            out = reward_model(**enc)
        store.append(out.logits.squeeze().item())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(chosen_scores,   bins=15, alpha=0.7, label="Chosen",   color="#4A90D9", edgecolor="white")
ax.hist(rejected_scores, bins=15, alpha=0.7, label="Rejected", color="#E07B54", edgecolor="white")
ax.axvline(np.mean(chosen_scores),   color="#2C6FAD", linestyle="--", linewidth=1.5, label=f"Chosen mean: {np.mean(chosen_scores):.3f}")
ax.axvline(np.mean(rejected_scores), color="#B85A38", linestyle="--", linewidth=1.5, label=f"Rejected mean: {np.mean(rejected_scores):.3f}")
ax.set_title("Reward Score Distribution (Eval Set)", fontsize=13, fontweight="bold")
ax.set_xlabel("Reward Score", fontsize=11)
ax.set_ylabel("Count", fontsize=11)
ax.legend(fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()

chart_path = paths["results"] / "reward_score_distribution.png"
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {chart_path}")

In [ ]:
# Save reward model checkpoint and publish artifacts
import json

rm_checkpoint = paths["results"] / "reward_checkpoint"
trainer.save_model(str(rm_checkpoint))
tokenizer.save_pretrained(str(rm_checkpoint))
print(f"Reward model saved to {rm_checkpoint}")

log_history = trainer.state.log_history
metrics = {
    "base_model": REWARD_BASE,
    "num_train_pairs": len(train_dataset),
    "num_eval_pairs": len(eval_dataset),
    "chosen_mean_score": float(np.mean(chosen_scores)),
    "rejected_mean_score": float(np.mean(rejected_scores)),
    "log_history": log_history,
}

metrics_path = paths["results"] / "reward_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2))

publish_artifacts(
    [metrics_path, paths["results"] / "reward_score_distribution.png"],
    "add reward model metrics and score distribution chart",
    repo_root,
)
print("Done.")